In [9]:
# Imports
import pandas as pd

# Load the dataset
df = pd.read_csv('../data/phishing_url_dataset_raw.csv')
df.shape

(235795, 55)

In [10]:
# Keep only numeric columns, drop text ones
df_numeric = df.select_dtypes(include='number')
df_numeric.shape

(235795, 51)

In [11]:
# Drop five columns, to match expectations
df_numeric = df_numeric.drop(['DomainTitleMatchScore', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'HasObfuscation'], axis=1)
df_numeric.shape

(235795, 46)

In [13]:
# Split the data into inputs (x, everything except the label) and the answer key (y, the label)
# Then divide both into 80 % training and 20% test set, keeping the same phishing/legitimate ration in both (stratify=y)
X = df_numeric.drop('label', axis=1)
y = df_numeric['label']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [14]:
# Confirm the split
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((188636, 45), (47159, 45), (188636,), (47159,))

In [15]:
# Train a Random Forest 5 seperate times on different slices of training data (cross-validation)
# Then print average accuracy and how much it varies across those 5 runs
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')
print(f"Random Forest CV accuracy: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})")

Random Forest CV accuracy: 1.0000 (+/- 0.0000)


In [16]:
# Check for data leakage causing 100% score - which is suspicious
# Check for duplicate rows and any feature too perfectly correlated with the label
df_numeric.duplicated().sum()
df_numeric.corr()['label'].sort_values(ascending=False)

label                         1.000000
URLSimilarityIndex            0.860358
HasSocialNet                  0.784255
HasCopyrightInfo              0.743358
HasDescription                0.690232
IsHTTPS                       0.609132
HasSubmitButton               0.578561
IsResponsive                  0.548608
URLTitleMatchScore            0.539419
HasHiddenFields               0.507731
HasFavicon                    0.493711
URLCharProb                   0.469749
CharContinuationRate          0.467735
HasTitle                      0.459725
Robots                        0.392620
NoOfJS                        0.373500
Pay                           0.359747
NoOfSelfRef                   0.316211
NoOfImage                     0.274658
LineOfCode                    0.272257
NoOfExternalRef               0.258627
NoOfiFrame                    0.225822
Bank                          0.188959
HasExternalFormSubmit         0.167574
HasPasswordField              0.138183
NoOfEmptyRef             

In [18]:
# Check how many rows in dataset are exact duplicates of another row
# If it has a high number, it could explain why the model scored a suspicious 100%
df_numeric.duplicated().sum()

np.int64(808)

In [19]:
# The results were 808 duplicates out of 235,795 rows which is only 0.34%
# We will run one more diagnostic to figure out what is happening by training a weak model, Logistic Regression.
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

lr = LogisticRegression(max_iter=1000)
lr_scores = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"Logistic Regression CV accuracy: {lr_scores.mean():.4f} (+/- {lr_scores.std():.4f})")

Logistic Regression CV accuracy: 0.9999 (+/- 0.0000)


In [20]:
# RF got 100% - suspicious, check leakage later


from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_scores = cross_val_score(xgb, X_train, y_train, cv=5, scoring='accuracy')
print(f"XGBoost CV accuracy: {xgb_scores.mean():.4f} (+/- {xgb_scores.std():.4f})")

XGBoost CV accuracy: 1.0000 (+/- 0.0000)
